# **資產負債表**

In [ ]:
import requests
import pandas as pd
import os
import time

# === 年度與季度 ===
year = "113"
season = "4"

# === 設定自動輸出資料夾 ===
folder_name = f"{year}Q{season}"
# 假設你希望將輸出資料夾放在專案的 'output' 子資料夾中
output_dir = os.path.join("output", folder_name)
os.makedirs(output_dir, exist_ok=True)

# 讀取 Excel
excel_file_path = 'stockcode.xlsx'
df = pd.read_excel(excel_file_path, sheet_name='財報_202504')

# 確保所有代碼是字串格式
all_codes = df['代號'].astype(str)  
stock_ids = set(all_codes)

url = "https://mops.twse.com.tw/mops/api/t164sb03"

headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://mops.twse.com.tw/mops/web/t164sb03"
}

# 紀錄失敗的代碼
error_stock_ids = []

In [ ]:
# 主程式
for stock_id in stock_ids:
    payload = {
        "companyId": stock_id,
        "dataType": "1",  # 資產負債表
        "year": year,
        "season": season,
        "subsidiaryCompanyId": ""
    }

    try:
        resp = requests.post(url, json=payload, headers=headers)
        resp.raise_for_status()

        if not resp.text.strip():
            print(f"❌ 錯誤：{stock_id} 回傳空白（可能被擋）")
            continue

        try:
            data = resp.json()
        except ValueError:
            print(f"❌ 錯誤：{stock_id} 回傳非 JSON：{resp.text[:100]}")
            continue

        if data.get("code") != 200 or "reportList" not in data["result"]:
            print(f"⚠️ 無資料：{stock_id}")
            continue

        rows = data["result"]["reportList"]

        # === 自動設定欄位名稱 ===
        if season == "4":
            col_names = [
                "會計項目",
                f"{year}年Q{season}金額",
                f"{year}年Q{season}占比%",
                f"{int(year)-1}年Q{season}金額",
                f"{int(year)-1}年Q{season}占比%"
            ]
        else:
            col_names = [
                "會計項目",
                f"{year}年Q{season}金額",
                f"{year}年Q{season}占比%",
                f"{int(year)-1}年Q4金額",
                f"{int(year)-1}年Q4占比%",
                f"{int(year)-1}年Q{season}金額",
                f"{int(year)-1}年Q{season}占比%"
            ]

        # 建立 DataFrame
        df = pd.DataFrame(rows, columns=col_names)

        # 儲存檔案
        filename = f"{year}Q{season}_{stock_id}_資產負債表.xlsx"
        filepath = os.path.join(output_dir, filename)
        df.to_excel(filepath, index=False)
        print(f"✅ 儲存完成：{filename}")

    except Exception as e:
        print(f"❌ 例外錯誤：{stock_id}: {e}")
        error_stock_ids.append(stock_id)

    time.sleep(0.3)

if error_stock_ids:
    with open("failed_資產負債表.txt", "w", encoding="utf-8") as f:
        for code in error_stock_ids:
            f.write(f"{code}\n")
    print(f"\n❌ 共 {len(error_stock_ids)} 筆失敗，已寫入 failed_資產負債表.txt")
else:
    print("\n🎉 全部成功，沒有錯誤！")

In [ ]:
if error_stock_ids:
    with open("failed_資產負債表.txt", "w", encoding="utf-8") as f:
        for code in error_stock_ids:
            f.write(f"{code}\n")
    print(f"\n❌ 共 {len(error_stock_ids)} 筆失敗，已寫入 failed_資產負債表.txt")
else:
    print("\n🎉 全部成功，沒有錯誤！")

### **<font color = ORANGE>爬取failed_資產負債表<font>**

In [ ]:
import requests
import pandas as pd
import os
import time

# === 年度與季度 ===
year = "113"
season = "4"

# === 設定輸出路徑（與原本相同即可）===
folder_name = f"{year}Q{season}"
output_dir = fr"./data/processed/balance_sheets/{folder_name}"
os.makedirs(output_dir, exist_ok=True)

# === 讀取失敗清單 ===
with open("failed_資產負債表.txt", "r", encoding="utf-8") as f:
    retry_ids = [line.strip() for line in f if line.strip()]

print(f"🔁 共有 {len(retry_ids)} 筆準備重試...")

# API 設定
url = "https://mops.twse.com.tw/mops/api/t164sb03"
headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://mops.twse.com.tw/mops/web/t164sb03"
}


# 記錄重新失敗的代碼
retry_failed_ids = []

# === 開始重試 ===
for stock_id in retry_ids:
    payload = {
        "companyId": stock_id,
        "dataType": "1",  # 資產負債表
        "year": year,
        "season": season,
        "subsidiaryCompanyId": ""
    }

    try:
        resp = requests.post(url, json=payload, headers=headers)
        resp.raise_for_status()

        if not resp.text.strip():
            print(f"❌ 回傳空白：{stock_id}")
            retry_failed_ids.append(stock_id)
            continue

        try:
            data = resp.json()
        except ValueError:
            print(f"❌ 回傳非 JSON：{stock_id}")
            retry_failed_ids.append(stock_id)
            continue

        if data.get("code") != 200 or "reportList" not in data["result"]:
            print(f"⚠️ 無資料：{stock_id}")
            retry_failed_ids.append(stock_id)
            continue

        rows = data["result"]["reportList"]

        # === 自動調整欄位名稱 ===
        if season == "4":
            col_names = [
                "會計項目",
                f"{year}年金額",
                f"{year}年占比%",
                f"{int(year)-1}年金額",
                f"{int(year)-1}年占比%"
            ]
        else:
            col_names = [
                "會計項目",
                f"{year}年Q{season}金額",
                f"{year}年Q{season}占比%",
                f"{int(year)-1}年Q4金額",
                f"{int(year)-1}年Q4占比%",
                f"{int(year)-1}年Q{season}金額",
                f"{int(year)-1}年Q{season}占比%"
            ]

        try:
            df = pd.DataFrame(rows, columns=col_names)
        except Exception as e:
            print(f"❌ DataFrame 例外錯誤：{stock_id}: {e}")
            retry_failed_ids.append(stock_id)
            continue

        filename = f"{year}Q{season}_{stock_id}_資產負債表.xlsx"
        filepath = os.path.join(output_dir, filename)
        df.to_excel(filepath, index=False)
        print(f"✅ 重試成功：{filename}")

    except Exception as e:
        print(f"❌ 例外錯誤：{stock_id}: {e}")
        retry_failed_ids.append(stock_id)

    time.sleep(0.3)

# === 寫入最終還是失敗的 ===
if retry_failed_ids:
    with open("failed_資產負債表.txt", "w", encoding="utf-8") as f:
        for code in retry_failed_ids:
            f.write(f"{code}\n")
    print(f"\n❌ 還有 {len(retry_failed_ids)} 筆重試失敗，已存為 failed_資產負債表.txt")
else:
    print("\n🎉 重試全部成功！")


# **綜合損益表**

In [ ]:
import requests
import pandas as pd
import os
import time

# === 年度與季度 ===
year = "113"
season = "4"

# === 設定自動輸出資料夾 ===
folder_name = f"{year}年損益表"
output_dir = fr"./data/processed/Income Statement/{folder_name}"
os.makedirs(output_dir, exist_ok=True)

# 讀取 Excel
excel_file_path = 'stockcode.xlsx'
df = pd.read_excel(excel_file_path, sheet_name='財報_202504')

# 確保所有代碼是字串格式
all_codes = df['代號'].astype(str)  
stock_ids = set(all_codes)

url = "https://mops.twse.com.tw/mops/api/t164sb04"

headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://mops.twse.com.tw/mops/web/t164sb04"
}

# 紀錄失敗的代碼
error_stock_ids = []

In [ ]:
for stock_id in stock_ids:
    payload = {
        "companyId": stock_id,
        "dataType": "1",  # 損益表
        "year": year,
        "season": season,
        "subsidiaryCompanyId": ""
    }

    try:
        resp = requests.post(url, json=payload, headers=headers)
        resp.raise_for_status()

        if not resp.text.strip():
            print(f"❌ 回傳空白：{stock_id}")
            error_stock_ids.append(stock_id)
            continue

        try:
            data = resp.json()
        except ValueError:
            print(f"❌ 回傳非 JSON：{stock_id}")
            error_stock_ids.append(stock_id)
            continue

        if data.get("code") != 200 or "reportList" not in data["result"]:
            print(f"⚠️ 無有效資料：{stock_id}")
            error_stock_ids.append(stock_id)
            continue

        rows = data["result"]["reportList"]
        
        # 嘗試建立 DataFrame
        try:
            df = pd.DataFrame(
                rows,
                columns = [
                "會計項目",
                f"{year}年金額",
                f"{year}年占比%",
                f"{int(year)-1}年金額",
                f"{int(year)-1}年占比%"
            ]
            )
        except Exception as e:
            print(f"❌ 例外錯誤：{stock_id}: {e}")
            error_stock_ids.append(stock_id)
            continue

        filename = f"{year}Q{season}_{stock_id}_損益表.xlsx"
        filepath = os.path.join(output_dir, filename)
        df.to_excel(filepath, index=False)
        print(f"✅ 儲存完成：{filename}")

    except Exception as e:
        print(f"❌ 例外錯誤：{stock_id}: {e}")
        error_stock_ids.append(stock_id)

    time.sleep(0.3)

In [ ]:
if error_stock_ids:
    with open("failed_損益表.txt", "w", encoding="utf-8") as f:
        for code in error_stock_ids:
            f.write(f"{code}\n")
    print(f"\n❌ 共 {len(error_stock_ids)} 筆失敗，已寫入 failed_損益表.txt")
else:
    print("\n🎉 全部成功，沒有錯誤！")

### **<font color = ORANGE>爬取failed_損益表<font>**

In [ ]:
import requests
import pandas as pd
import os
import time

# === 年度與季度 ===
year = "113"
season = "4"

# === 設定輸出路徑（與原本相同即可）===
folder_name = f"{year}年損益表"
output_dir = fr"./data/processed/Income Statement/{folder_name}"
os.makedirs(output_dir, exist_ok=True)

# === 讀取 failed.txt 失敗清單 ===
with open("failed_損益表.txt", "r", encoding="utf-8") as f:
    retry_ids = [line.strip() for line in f if line.strip()]

print(f"🔁 共有 {len(retry_ids)} 筆準備重試...")

# API 設定
url = "https://mops.twse.com.tw/mops/api/t164sb04"
headers = {
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://mops.twse.com.tw/mops/web/t164sb04"
}

# 記錄重新失敗的代碼
retry_failed_ids = []

# === 開始重試 ===
for stock_id in retry_ids:
    payload = {
        "companyId": stock_id,
        "dataType": "2",  # 損益表
        "year": year,
        "season": season,
        "subsidiaryCompanyId": ""
    }

    try:
        resp = requests.post(url, json=payload, headers=headers)
        resp.raise_for_status()

        if not resp.text.strip():
            print(f"❌ 回傳空白：{stock_id}")
            retry_failed_ids.append(stock_id)
            continue

        try:
            data = resp.json()
        except ValueError:
            print(f"❌ 回傳非 JSON：{stock_id}")
            retry_failed_ids.append(stock_id)
            continue

        if data.get("code") != 200 or "reportList" not in data["result"]:
            print(f"⚠️ 無資料：{stock_id}")
            retry_failed_ids.append(stock_id)
            continue

        rows = data["result"]["reportList"]
        
        try:
            df = pd.DataFrame(
                rows,
                columns = [
                "會計項目",
                f"{year}年金額",
                f"{year}年占比%",
                f"{int(year)-1}年金額",
                f"{int(year)-1}年占比%"
            ]
            )
        except Exception as e:
            print(f"❌ DataFrame 例外錯誤：{stock_id}: {e}")
            retry_failed_ids.append(stock_id)
            continue

        filename = f"{year}_{stock_id}_損益表.xlsx"
        filepath = os.path.join(output_dir, filename)
        df.to_excel(filepath, index=False)
        print(f"✅ 重試成功：{filename}")

    except Exception as e:
        print(f"❌ 例外錯誤：{stock_id}: {e}")
        retry_failed_ids.append(stock_id)

    time.sleep(0.3)

# === 寫入最終還是失敗的 ===
if retry_failed_ids:
    with open("failed_損益表.txt", "w", encoding="utf-8") as f:
        for code in retry_failed_ids:
            f.write(f"{code}\n")
    print(f"\n❌ 還有 {len(retry_failed_ids)} 筆重試失敗，已存為 failed_損益表.txt")
else:
    print("\n🎉 重試全部成功！")


# **其他檢查用**

In [ ]:
import os
import pandas as pd

# === 1. 讀取原始代碼（Excel 第2欄 B欄）===
df = pd.read_excel("stockcode.xlsx", sheet_name="財報_202504")  # ← 修改成實際檔名與工作表
original_ids = df.iloc[:, 1].dropna().astype(int).astype(str).str.zfill(4).tolist()
original_set = set(original_ids)

# === 2. 從產出的損益表檔名中取得成功代碼 ===
report_folder = r"./data/processed/Income Statement/{folder_name}"  # ← 修改為實際路徑
success_ids = []
for fname in os.listdir(report_folder):
    if fname.endswith(".xlsx") and "損益表" in fname:
        parts = fname.split("_")
        if len(parts) >= 2:
            success_ids.append(parts[1].zfill(4))
success_set = set(success_ids)

# === 3. 比對找出「沒成功產出的代碼」===
missing_ids = original_set - success_set

# === 4. 輸出結果 ===
print(f"\n🟡 共有 {len(missing_ids)} 筆原始代碼沒有產出 Excel：")
print(sorted(missing_ids))

# === 5. 儲存成 missing_only.txt ===
with open("missing_only.txt", "w", encoding="utf-8") as f:
    for sid in sorted(missing_ids):
        f.write(f"{sid}\n")

print("✅ 已儲存為 missing_only.txt")


In [ ]:
import os
import pandas as pd

# 資料夾路徑（請確認這是你實際的路徑）
folder_path = r"./data/processed/Income Statement/{folder_name}"

# 建立一個列表儲存股票代碼
stock_ids = []

# 讀取檔案名稱並擷取代碼
for fname in os.listdir(folder_path):
    if fname.endswith(".xlsx") and "損益表" in fname:
        parts = fname.split("_")
        if len(parts) >= 2:
            stock_id = parts[1].zfill(4)  # 保險補滿4碼
            stock_ids.append(stock_id)

# 建立 DataFrame 並輸出 Excel
df = pd.DataFrame(stock_ids, columns=["股票代碼"])
output_file = "extracted_stock_ids.xlsx"
df.to_excel(output_file, index=False)

print(f"✅ 已成功擷取 {len(stock_ids)} 筆股票代碼，並存成 {output_file}")


In [ ]:
import os
import pandas as pd

# === 1. 讀取原始代碼（從 stockcode.xlsx 的 B欄）===
df = pd.read_excel("stockcode.xlsx", sheet_name="財報_202504")  # ← 修改為你 Excel 的實際名稱
all_ids = df.iloc[:, 1].dropna().astype(int).astype(str).str.zfill(4).tolist()
all_set = set(all_ids)

# === 2. 讀取已產出的資產負債表檔案代碼 ===
folder_path = r"./data/processed/balance_sheets/{folder_name}"
success_ids = []
for fname in os.listdir(folder_path):
    if fname.endswith(".xlsx") and "資產負債表" in fname:
        parts = fname.split("_")
        if len(parts) >= 2:
            stock_id = parts[1].zfill(4)
            success_ids.append(stock_id)
success_set = set(success_ids)

# === 3. 計算未抓到的代碼 ===
missing_ids = sorted(all_set - success_set)

# === 4. 輸出為 TXT 檔 ===
with open("missing_balance_ids.txt", "w", encoding="utf-8") as f:
    for sid in missing_ids:
        f.write(f"{sid}\n")

# === 5. 顯示結果 ===
print(f"\n🔍 共 {len(missing_ids)} 筆原始代碼未產出資產負債表 Excel：")
print(missing_ids[:10], "...")  # 顯示前幾筆預覽
print("✅ 已儲存為 missing_balance_ids.txt")
